# `q_diagonalize()`

The analysis function `nematics3d.q_diagonalize()` converts a symmetric, traceless $Q$-tensor into the scalar order parameter $S$ and its principal director $\mathbf{n}$. It can also return the complete eigensystem when all three eigenvalues and eigenvectors are needed.

The function supports parallel execution through either the compiled C backend or the `NumExpr` backend. The final section benchmarks 1, 2, and 4 workers on the included example field, providing a preliminary test for choosing a suitable worker count on your own machine. The fastest setting depends on the field size, processor, memory bandwidth, and software environment. See [Details](#Details) for an overview of the specialized diagonalization algorithm.

**Negative $S$ can be physically meaningful for oblate or anti-nematic ordering, including some discotic liquid-crystal systems. This convention is not currently supported because this function defines $S=3\lambda_{\max}/2$ from the largest eigenvalue of $Q$.**


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy` and `Nematics3D`; no setup detail is needed to understand the examples.


In [1]:
import numpy as np
import nematics3d as n3d

## Inputs and outputs

The public signature is:

```python
q_diagonalize(
    qtensor,
    *,
    is_biaxial=False,
    is_right_handed=False,
    is_use_c_backend=None,
    worker_count=None,
)
```

### Accepted $Q$-tensor representations

`qtensor` must be a non-empty floating-point `NumPy` array in one of two representations:

| Representation | Trailing shape | Stored components |
| --- | --- | --- |
| Compact `QField5` | `(..., 5)` | `(Q_xx, Q_xy, Q_xz, Q_yy, Q_yz)` |
| Full `QField9` | `(..., 3, 3)` | the complete matrix $Q$ |

The leading dimensions are preserved, so the same function accepts one tensor, a list of tensors, a two-dimensional slice, or a three-dimensional field. Empty inputs, integer arrays, `NaN`, and infinite values are rejected. Full $3\times3$ input is additionally checked for symmetry and zero trace. Compact input is symmetric and traceless by construction, with $Q_{zz}=-Q_{xx}-Q_{yy}$.

### Options

| Argument | Meaning |
| --- | --- |
| `is_biaxial=False` | Compute only the largest eigenvalue and its director. This is the default and lowest-memory path. |
| `is_biaxial=True` | Also return all eigenvalues and eigenvectors, ordered by descending eigenvalue. |
| `is_right_handed=False` | Leave the complete eigenvector frame in the orientation produced by the solver. |
| `is_right_handed=True` | Make every complete eigenvector frame right-handed. This requires `is_biaxial=True`. |
| `is_use_c_backend=None` | Prefer the compiled C backend and fall back to `NumExpr` if the extension is unavailable. |
| `is_use_c_backend=True` | Require the compiled C backend; raise `ImportError` if it is unavailable. |
| `is_use_c_backend=False` | Force the `NumExpr` backend. |
| `worker_count=None` | Use the selected backend's automatic worker behavior. |
| `worker_count=1, 2, ...` | Request an explicit number of C workers or `NumExpr` threads. |

### Returned result

The function returns a `QDiagonalizationResult` with these fields:

| Field | Default mode | Complete mode | Meaning |
| --- | --- | --- | --- |
| `S` | array | array | $S=3\lambda_{\max}/2$ |
| `n` | array | array | unit eigenvector belonging to $\lambda_{\max}$ |
| `isotropic_indices` | list | list | coordinates where the deterministic isotropic convention was applied |
| `eigenvalues` | `None` | array | all three eigenvalues in descending order |
| `eigenvectors` | `None` | array | matching eigenvectors stored in columns |

The shape of `S` is the leading shape of `qtensor`, and `n` adds a final axis of length 3. Each entry in `isotropic_indices` is a coordinate tuple into those leading dimensions; a single tensor is represented by the empty coordinate `()`. Complete `eigenvalues` and `eigenvectors` add trailing shapes `(3,)` and `(3, 3)`, respectively.


## Examples


### Minimal example: recover $S$ and $\mathbf{n}$

For a positive uniaxial state,

$$Q=S\left(\mathbf{n}\mathbf{n}-\frac{I}{3}\right).$$

The default mode returns $S=3\lambda_{\max}/2$ and the unit eigenvector belonging to $\lambda_{\max}$.


In [2]:
S_input = 0.75
n_input = np.array([1.0, 1.0, 1.0])
n_input /= np.linalg.norm(n_input)
Q = S_input * (np.outer(n_input, n_input) - np.eye(3) / 3.0)

result = n3d.q_diagonalize(Q)
print("S =", result.S)
print("n =", result.n)
print("axis overlap =", abs(np.dot(result.n, n_input)))

S = 0.7500000000000002
n = [0.57735027 0.57735027 0.57735027]
axis overlap = 1.0


The sign of an eigenvector is arbitrary. Compare nematic axes with $|\mathbf n_1\cdot\mathbf n_2|$, not by requiring componentwise equality.

### Compact five-component input

Use compact input when the symmetry and tracelessness of $Q$ are already guaranteed. It stores five values per tensor instead of nine.


In [3]:
Q5 = np.array([Q[0, 0], Q[0, 1], Q[0, 2], Q[1, 1], Q[1, 2]])
compact_result = n3d.q_diagonalize(Q5)
print("same S:", np.allclose(compact_result.S, result.S))
print("same axis:", abs(np.dot(compact_result.n, result.n)))

same S: True
same axis: 0.9999999999999999


### A field or batch of tensors

Leading dimensions are treated as batch or spatial axes and are preserved in the returned arrays.


In [4]:
Q_batch = np.stack([Q, 0.5 * Q])
batch_result = n3d.q_diagonalize(Q_batch)
print("input:", Q_batch.shape)
print("S:", batch_result.S.shape, batch_result.S)
print("n:", batch_result.n.shape)

input: (2, 3, 3)
S: (2,) [0.75  0.375]
n: (2, 3)


### Selecting a backend and worker count

Most users should leave both `is_use_c_backend` and `worker_count` at `None`. Explicit selection is useful for reproducible comparisons or for confirming that a wheel contains the compiled extension. A forced C request raises `ImportError` rather than silently changing algorithms when the extension is unavailable.


In [5]:
automatic = n3d.q_diagonalize(Q_batch)
c_single_thread = n3d.q_diagonalize(
    Q_batch,
    is_use_c_backend=True,
    worker_count=1,
)
python_two_threads = n3d.q_diagonalize(
    Q_batch,
    is_use_c_backend=False,
    worker_count=2,
)

print("automatic and C S agree:", np.allclose(automatic.S, c_single_thread.S))
print(
    "C and NumExpr axes agree:",
    np.allclose(np.abs(np.sum(c_single_thread.n * python_two_threads.n, axis=-1)), 1.0),
)


automatic and C S agree: True
C and NumExpr axes agree: True


### Complete eigensystem

Set `is_biaxial=True` to return descending eigenvalues and matching eigenvector columns. Set `is_right_handed=True` only when downstream work requires a consistently oriented three-vector frame.


In [6]:
axes, _ = np.linalg.qr(np.random.default_rng(7).normal(size=(3, 3)))
expected_values = np.array([0.6, -0.1, -0.5])
Q_biaxial = axes @ np.diag(expected_values) @ axes.T

complete = n3d.q_diagonalize(
    Q_biaxial,
    is_biaxial=True,
    is_right_handed=True,
)
Q_reconstructed = np.einsum(
    "...ik,...k,...jk->...ij",
    complete.eigenvectors,
    complete.eigenvalues,
    complete.eigenvectors,
)
print("eigenvalues =", complete.eigenvalues)
print("S =", complete.S)
print("right-handed determinant =", np.linalg.det(complete.eigenvectors))
print("reconstruction error =", np.max(np.abs(Q_reconstructed - Q_biaxial)))

eigenvalues = [ 0.6 -0.1 -0.5]
S = 0.8999999999999999
right-handed determinant = 0.9999999999999998
reconstruction error = 3.3306690738754696e-16


## Special examples


### The isotropic case

At an isotropic point, all three eigenvalues are equal and no physical director is defined. `Nematics3D` uses the deterministic convention $S=0$ and $\mathbf{n}=(1,0,0)$, and records the affected coordinates in `isotropic_indices`. Complete mode uses the identity matrix as the placeholder eigenvector frame.


In [7]:
isotropic = n3d.q_diagonalize(np.zeros((3, 3)))
print("S =", isotropic.S)
print("placeholder n =", isotropic.n)
print("isotropic indices =", isotropic.isotropic_indices)

[WARNING]
    <q_diagonalize> 
    >>> 1 near-isotropic grid point(s). Set S to 0 and assigned the default director [1, 0, 0] at those points. Inspect result.isotropic_indices before interpreting the director.
    Current warning call: D:\Document\GitHub\Nematics3D\src\nematics3d\analysis\q_diagonalization\_solver.py:620
    Caller: C:\Users\myy23\AppData\Local\Temp\ipykernel_23012\3532640433.py:1
    code: isotropic = n3d.q_diagonalize(np.zeros((3, 3)))


S = 0.0
placeholder n = [1. 0. 0.]
isotropic indices = [()]


## Details

### A solver specialized for $Q$ tensors

A symmetric, traceless $3\times3$ tensor has only five independent components. `q_diagonalize()` uses this structure directly instead of treating every input as an arbitrary dense matrix. The eigenvalues are obtained from the trigonometric solution of the traceless characteristic cubic. In the default principal-only mode, the solver computes only the largest eigenvalue and its eigenvector, because those are all that are needed for $S$ and $\mathbf{n}$.

For readers interested specifically in the uniaxial use case, the principal-eigenpair calculation follows a classical analytic approach. The dominant eigenvalue is computed using the analytic traceless-$3\times3$ formulation described by [the note of Peterson (2019)](../../docs/reference/order_parameter_calculation.pdf), while the corresponding eigenvector is obtained using a robust maximal-norm adjugate construction. For an eigenvalue $\lambda$, this selects the strongest row of the adjugate of $Q-\lambda I$ instead of relying on a row that has become numerically tiny.

### Constructing the complete eigensystem

The solver follows the robust analytic approach to the symmetric $3\times3$ eigenproblem developed in earlier work, particularly [Scherzinger and Dohrmann (2008)](https://www.sciencedirect.com/science/article/abs/pii/S0045782508001436), but is specialized here to traceless $Q$ tensors and uses a distinct construction for the eigensystem.

Complete mode first identifies the eigenvalue that is separated from the other two and computes its eigenvector. It then constructs an orthonormal basis for the perpendicular plane, projects $Q$ into that two-dimensional plane, and diagonalizes the resulting symmetric $2\times2$ block with a stable rotation. The three eigenvectors are therefore produced as one orthonormal frame.

A more direct analytic implementation could recover all three eigenvectors independently from separate adjugates or cross products. Near a repeated eigenvalue, however, those vectors can become very small, noisy, or almost parallel. The isolated-eigenpair and projected-plane design avoids that failure mode. If two eigenvalues are exactly equal, individual axes inside their plane remain physically non-unique, but the returned frame stays orthonormal.

### Why it can be faster than `NumPy`

`numpy.linalg.eigh()` is a robust general-purpose interface for symmetric or Hermitian matrices. It must support matrix sizes and layouts far beyond this fixed $3\times3$, symmetric, traceless case. In contrast, `q_diagonalize()` eliminates the redundant tensor component, uses fixed scalar formulas, avoids general eigensolver setup, and does not compute the two unused eigenpairs in principal-only mode.

The compiled C backend processes contiguous tensor fields in tight loops, releases `Python`'s global interpreter lock, and can divide the field among `Python` worker threads. The `NumExpr` backend evaluates the same array-oriented stages in compiled loops and can also use multiple threads. These advantages matter most for fields containing many small $Q$ tensors; for a single tensor, dispatch overhead can dominate and the performance difference is not important. The benchmark below measures the trade-off on the bundled example field.


## Possible issues

### Negative $S$

Negative $S$ can be physically meaningful for oblate or anti-nematic ordering, including some discotic liquid-crystal systems. This convention is not currently supported because the function defines $S=3\lambda_{\max}/2$ from the largest eigenvalue of $Q$.

### Isotropic points

A physical director is undefined at an isotropic point. The returned deterministic placeholder must not be interpreted as a real molecular orientation; use `isotropic_indices` to identify and handle these locations.

### Degenerate or nearly degenerate eigenvalues

When two eigenvalues are equal or very close, their individual eigenvectors are non-unique or sensitive to small numerical perturbations. The subspace spanned by those eigenvectors can still be correct even when the individual axes differ substantially between calculations.

### The sign of the director

The directors $\mathbf{n}$ and $-\mathbf{n}$ represent the same nematic orientation. Compare axes using $|\mathbf{n}_1\mathbin{\cdot}\mathbf{n}_2|$ rather than componentwise equality. This function does not apply a special convention to select one of these two directions, and its designers do not intend to prescribe a particular sign for $\mathbf{n}$. The right-handed option only fixes the handedness of a complete eigenvector frame; it does not assign a unique sign to an individual director.


## Logging and progress information

Near-isotropic tensors produce a warning at the default log level. To inspect stage-by-stage progress for other inputs, set `log_level=logging.DEBUG`. The logger then reports the input summary, selected solver, major calculation stages, and their elapsed times. The following example uses `Q_example_workflow.npy`; timings will vary by machine.


In [8]:
import logging
from pathlib import Path


def find_example_data():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        path = candidate / "example/data/Q_example_workflow.npy"
        if path.exists():
            return path
    raise FileNotFoundError("Run this tutorial from inside the Nematics3D repository.")


Q_example = np.load(find_example_data())
debug_result = n3d.q_diagonalize(
    Q_example,
    worker_count=1,
    log_level=logging.DEBUG,
)


[DEBUG]
    Function `q_diagonalize` STARTED in program `ipykernel_launcher.py`
[DEBUG]
    <q_diagonalize> 
    Received Q-tensor input: shape=(200, 100, 100, 5), dtype=float32.
    Options: is_biaxial=False, is_right_handed=False, is_use_c_backend=None, worker_count=1.
[DEBUG]
    <q_diagonalize> 
    Preparing and validating Q-tensor input.


[DEBUG]
    <q_diagonalize> 
    Prepared 2,000,000 Q tensor(s) from compact five-component input in 0.005 seconds.
[DEBUG]
    <q_diagonalize> 
    Selected compiled C principal-eigenpair solver with 1 worker(s).
[DEBUG]
    <q_diagonalize> 
    Classifying 2,000,000 Q tensor(s) for near isotropy.
[DEBUG]
    <q_diagonalize> 
    Classified 2,000,000 Q tensor(s) in 0.061 seconds: 0 near-isotropic and 2,000,000 non-isotropic.
[DEBUG]
    <q_diagonalize> 
    Computing 2,000,000 principal-eigenpair result(s) with the compiled C solver.
[DEBUG]
    <q_diagonalize> 
    Computed 2,000,000 principal-eigenpair result(s) in 0.060 seconds.
[DEBUG]
    <q_diagonalize> 
    Finalizing diagonalization results.
[DEBUG]
    <q_diagonalize> 
    Finalized diagonalization results in 0.005 seconds.
[DEBUG]
    Function `q_diagonalize` FINISHED in program `ipykernel_launcher.py`. Elapsed time: 0.131 seconds.


## Performance benchmarks

The benchmarks below use `example/data/Q_example_workflow.npy` and launch a fresh subprocess for every configuration. This ensures that `NumExpr`, `BLAS`, and the C backend see the requested thread limits before numerical libraries initialize.

Each operation is warmed up and then measured three times. Reported time is the median. `Peak MiB` is incremental allocation observed by `tracemalloc`, not total process RSS. Input preparation and correctness checks are outside the timed region.

Run Benchmark 1 first as a preliminary worker-count recommendation for the common principal-only path. Choose the fastest measured setting rather than assuming that more workers are always better. Benchmark 2 repeats the comparison for the larger complete-eigensystem output.

`worker_count` means software workers or threads; the operating system decides which physical or logical CPU cores execute them.


In [9]:
import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "example/data/Q_example_workflow.npy").exists():
            return candidate
    raise FileNotFoundError("Run this tutorial from inside the Nematics3D repository.")


repo_root = find_repo_root()
q5_path = repo_root / "example/data/Q_example_workflow.npy"

benchmark_program = textwrap.dedent(
    r"""
    import gc
    import json
    import os
    import platform
    import statistics
    import sys
    import time
    import tracemalloc

    import numexpr as ne
    import numpy as np
    import psutil

    sys.path.insert(0, sys.argv[4])
    from nematics3d import q_diagonalize
    from nematics3d.datatypes import as_qfield9

    backend = sys.argv[1]
    mode = sys.argv[2]
    workers = int(sys.argv[3])
    q5 = np.load(sys.argv[5])
    is_biaxial = mode == "complete"

    def measure(operation, repeats=3):
        times = []
        peaks = []
        result = None
        for _ in range(repeats):
            gc.collect()
            tracemalloc.start()
            start = time.perf_counter()
            result = operation()
            times.append(time.perf_counter() - start)
            peaks.append(tracemalloc.get_traced_memory()[1] / 1024**2)
            tracemalloc.stop()
        return result, statistics.median(times), max(peaks)

    if backend == "numpy":
        q9 = np.asarray(as_qfield9(q5, is_strict_3d_field=False), dtype=np.float64)
        q9[..., 2, 2] = -q9[..., 0, 0] - q9[..., 1, 1]
        np.linalg.eigh(q9.reshape(-1, 3, 3)[:1000])
        result, seconds, peak_mib = measure(lambda: np.linalg.eigh(q9))
    else:
        use_c = backend == "c"
        q_diagonalize(
            q5.reshape(-1, 5)[:1000],
            is_biaxial=is_biaxial,
            is_use_c_backend=use_c,
            worker_count=workers,
        )
        result, seconds, peak_mib = measure(
            lambda: q_diagonalize(
                q5,
                is_biaxial=is_biaxial,
                is_use_c_backend=use_c,
                worker_count=workers,
            )
        )

    blas = np.show_config(mode="dicts").get("Build Dependencies", {}).get("blas", {})
    print(
        json.dumps(
            {
                "backend": backend,
                "mode": mode,
                "workers": workers,
                "seconds": seconds,
                "peak_mib": peak_mib,
                "os": platform.platform(),
                "cpu": platform.processor() or platform.machine(),
                "physical_cores": psutil.cpu_count(logical=False),
                "logical_cores": psutil.cpu_count(logical=True),
                "memory_gib": psutil.virtual_memory().total / 1024**3,
                "python": platform.python_version(),
                "numpy": np.__version__,
                "numexpr": ne.__version__,
                "blas": (blas.get("name", "unavailable") + " " + blas.get("version", "")).strip(),
                "field_shape": q5.shape,
                "field_dtype": str(q5.dtype),
            }
        )
    )
    """
)


def run_benchmark(backend, mode, workers):
    environment = os.environ.copy()
    thread_value = str(workers)
    environment.update(
        {
            "NUMEXPR_NUM_THREADS": thread_value,
            "OMP_NUM_THREADS": thread_value,
            "OPENBLAS_NUM_THREADS": thread_value,
            "MKL_NUM_THREADS": thread_value,
            "BLIS_NUM_THREADS": thread_value,
            "VECLIB_MAXIMUM_THREADS": thread_value,
        }
    )
    completed = subprocess.run(
        [
            sys.executable,
            "-c",
            benchmark_program,
            backend,
            mode,
            str(workers),
            str(repo_root / "src"),
            str(q5_path),
        ],
        check=False,
        capture_output=True,
        text=True,
        env=environment,
    )
    if completed.returncode:
        raise RuntimeError(completed.stdout + completed.stderr)
    return json.loads(completed.stdout.strip())


def print_results(results):
    first = results[0]
    print("CPU:", first["cpu"])
    print("cores:", first["physical_cores"], "physical,", first["logical_cores"], "logical")
    print("memory: %.1f GiB" % first["memory_gib"])
    print("software: Python %s, NumPy %s, NumExpr %s" % (first["python"], first["numpy"], first["numexpr"]))
    print("BLAS:", first["blas"])
    print("backend   workers   seconds   peak MiB")
    for result in results:
        print(
            "%-9s %7d %9.3f %10.1f"
            % (result["backend"], result["workers"], result["seconds"], result["peak_mib"])
        )


### Benchmark 1: principal-only C and `NumExpr` scaling

This measures the default `is_biaxial=False` output, including validation and result construction. It compares 1, 2, and 4 C workers with the same `NumExpr` thread counts.


In [10]:
principal_results = [
    run_benchmark(backend, "principal", workers)
    for backend in ("c", "numexpr")
    for workers in (1, 2, 4)
]
print_results(principal_results)


CPU: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
cores: 16 physical, 24 logical
memory: 79.7 GiB
software: Python 3.12.11, NumPy 2.3.2, NumExpr 2.14.2
BLAS: blas 3.9.0
backend   workers   seconds   peak MiB
c               1     0.137       63.0
c               2     0.107       63.0
c               4     0.093       63.0
numexpr         1     0.350      267.0
numexpr         2     0.274      267.0
numexpr         4     0.248      267.0


### Benchmark 2: complete eigensystem

This measures `is_biaxial=True` for C and `NumExpr` at 1, 2, and 4 workers. `numpy.linalg.eigh` is included at the same `BLAS` thread limits. `NumPy` returns ascending eigenvalues; `Nematics3D` returns descending eigenvalues.


In [11]:
complete_results = [
    run_benchmark(backend, "complete", workers)
    for backend in ("c", "numexpr", "numpy")
    for workers in (1, 2, 4)
]
print_results(complete_results)


CPU: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
cores: 16 physical, 24 logical
memory: 79.7 GiB
software: Python 3.12.11, NumPy 2.3.2, NumExpr 2.14.2
BLAS: blas 3.9.0
backend   workers   seconds   peak MiB
c               1     0.226      200.3
c               2     0.159      200.3
c               4     0.127      200.3
numexpr         1     0.825      345.3
numexpr         2     0.625      345.3
numexpr         4     0.547      345.3
numpy           1     2.508      183.1
numpy           2     2.514      183.1
numpy           4     2.502      183.1


Benchmark results depend on CPU frequency, memory bandwidth, operating-system load, wheel/compiler optimization, the `NumPy` build, and the `NumExpr` version. Keep the hardware and software report with quoted timings, and compare rows with the same worker limit.


## Where `q_diagonalize()` is used

Most users do not need to call this low-level function directly. `Nematics3D` uses it internally when higher-level objects or analyses need to recover $S$ and $\mathbf{n}$ from $Q$:

- [`QFieldObject.__init__()`](../classes/QFieldObject/initialize.ipynb) derives missing $S$ or $\mathbf{n}$ data from its stored $Q$ field;
- [`QPlane.act_refresh()`](../../src/nematics3d/classes/q_plane.py) and [`QSurface.act_refresh()`](../../src/nematics3d/classes/q_surface.py) update directors for sampled visual surfaces;
- [`defect_detect_surface()`](../../src/nematics3d/analysis/disclination/misc.py) obtains directors before surface-defect analysis;
- [`nml_principal_plane_analysis()`](../../src/nematics3d/principal_plane.py) diagonalizes sampled $Q$ tensors during principal-plane analysis.

Call `q_diagonalize()` directly when you have a standalone $Q$ array, need explicit backend or worker control, or require the complete eigensystem without constructing a higher-level object.


## Useful Links

### Referenced in this tutorial

- [QFieldObject initialization](../classes/QFieldObject/initialize.ipynb) — constructs a `QFieldObject` and derives missing $S$ or $\mathbf{n}$ from $Q$.
- [QFieldObject](../classes/QFieldObject/QFieldObject.ipynb) — overview of the main object that owns a $Q$-tensor field and its derived data.
- [QPlane](../classes/QPlane/QPlane.ipynb) — describes the plane object that refreshes sampled director data from $Q$.
- [`QPlane.act_refresh()` source](../../src/nematics3d/classes/q_plane.py) — shows the plane refresh path that obtains directors from $Q$.
- [`QSurface.act_refresh()` source](../../src/nematics3d/classes/q_surface.py) — shows the surface refresh path that obtains directors from $Q$.
- [`defect_detect_surface()` source](../../src/nematics3d/analysis/disclination/misc.py) — shows the surface-defect path that diagonalizes $Q$.
- [`nml_principal_plane_analysis()` source](../../src/nematics3d/principal_plane.py) — shows the principal-plane analysis path that diagonalizes sampled $Q$ tensors.
- [Peterson (2019)](../../docs/reference/order_parameter_calculation.pdf) — describes the analytic eigenvalues of a real symmetric $3\times3$ matrix used by the principal-eigenpair calculation.
- [Scherzinger and Dohrmann (2008)](https://www.sciencedirect.com/science/article/abs/pii/S0045782508001436) — develops a robust analytic solution for the symmetric $3\times3$ eigenproblem that motivates the complete solver.
- [`NumPy` `numpy.linalg.eigh`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigh.html) — reference implementation used for comparison in Benchmark 2.

### Going deeper

- [ResultBase](../classes/ResultBase/ResultBase.ipynb) — explains the named result-object interface returned by `q_diagonalize()`.
- [QFieldObject.act_defect_detect()](../classes/QFieldObject/act_defect_detect.ipynb) — detects defects after director data have been prepared.
- [QPlane.act_visualize_n()](../classes/QPlane/act_visualize_n.ipynb) — visualizes a sampled director field on a plane.
- [`as_qfield5()` and `as_qfield9()` source](../../src/nematics3d/datatypes/q_field.py) — defines the accepted compact and full $Q$-tensor representations and their validation rules.
